# Cross-Examination Arena GRPO Training

Credit-safe training notebook for Counsel-Env. Hugging Face GPU usage is disabled by default. Before running a remote training job, confirm the spend: a short A10G dry run is roughly $0.50, and a 90 minute A100 GRPO run is roughly $6-$10 depending on queue/runtime.

This notebook uses OpenEnv's `environment_factory` pattern, curriculum stages, shaped rewards from the environment, and rollout diagnostics saved to `rollout_debug.jsonl`.

In [ ]:
# Optional installs for Colab/HF Jobs. Keep commented until you approve credit/network usage.
# !pip install -U 'trl[vllm]' unsloth openenv-core datasets transformers accelerate matplotlib
# !pip install -e .

import json
import os
import random
import re
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

from counsel_env import CounselEnv
from counsel_env.models import CounselAction

MODEL = os.getenv('COUNSEL_MODEL', 'Qwen/Qwen3-1.7B')
ENV_URL = os.getenv('COUNSEL_ENV_URL', 'http://localhost:8000')
OUTPUT_DIR = os.getenv('COUNSEL_OUTPUT_DIR', './counsel-grpo-output')
RUN_TRAINING = os.getenv('RUN_TRAINING', '0') == '1'
MAX_STEPS = int(os.getenv('COUNSEL_MAX_STEPS', '200'))
DATASET_SIZE = int(os.getenv('COUNSEL_DATASET_SIZE', '256'))

print({'model': MODEL, 'env_url': ENV_URL, 'run_training': RUN_TRAINING, 'max_steps': MAX_STEPS, 'dataset_size': DATASET_SIZE})

In [ ]:
CURRICULUM_DISTRIBUTION = {
    'easy': {'easy': 1.0},
    'medium': {'easy': 0.25, 'medium': 0.75},
    'hard': {'easy': 0.10, 'medium': 0.30, 'hard': 0.60},
}

def sample_stage_schedule(total: int = 256) -> List[str]:
    # Warm start on easy, then medium, then hard/mixed challenge cases.
    schedule = []
    for i in range(total):
        frac = i / max(1, total - 1)
        if frac < 0.25:
            schedule.append('easy')
        elif frac < 0.70:
            schedule.append('medium')
        else:
            schedule.append('hard')
    return schedule

def create_dataset(num_samples: int = 256) -> Dataset:
    base_prompt = (
        'You are a sharp prosecuting attorney cross-examining a deterministic witness. '
        'Your goal is to surface contradictions by first making the witness commit to a claim, '
        'then presenting the exact exhibit that disproves it. Use the limited question budget efficiently. '
        'Avoid repeated, irrelevant, leading, or compound questions.'
    )
    stages = sample_stage_schedule(num_samples)
    prompts = [[{'role': 'user', 'content': base_prompt}] for _ in stages]
    return Dataset.from_dict({'prompt': prompts, 'curriculum_stage': stages})

dataset = create_dataset(DATASET_SIZE)
dataset[:3]

In [ ]:
def question_pattern(question: str) -> str:
    text = re.sub(r'[^a-z0-9\s]', ' ', (question or '').lower())
    return ' '.join(text.split()[:12])

class CounselToolEnv:
    """TRL environment wrapper. Constructor intentionally takes no args."""

    def __init__(self):
        self.client = CounselEnv(base_url=ENV_URL)
        self.reward = 0.0
        self.done = False
        self.stage = 'medium'
        self.questions: List[str] = []
        self.actions: List[str] = []
        self.last_components: Dict[str, Any] = {}

    def reset(self, **kwargs) -> str:
        """Reset the remote Counsel-Env episode."""
        self.stage = kwargs.get('curriculum_stage') or 'medium'
        result = self.client.reset(curriculum_stage=self.stage)
        obs = result.observation
        self.reward = 0.0
        self.done = False
        self.questions = []
        self.actions = []
        self.last_components = obs.reward_components
        return (
            f'CASE BRIEF:\n{obs.case_brief}\n\n'
            f'You have {obs.questions_remaining} questions. '
            f'Available exhibits: {obs.available_evidence}. '
            'Use ask_question, present_evidence, or rest_case.'
        )

    def ask_question(self, question: str) -> str:
        """Ask the witness a question.

        Args:
            question: The question to ask the witness.

        Returns:
            The witness response and remaining budget.
        """
        if self.done:
            raise ValueError('Episode is over.')
        self.actions.append('ask_question')
        self.questions.append(question)
        result = self.client.step(CounselAction(tool='ask_question', text=question))
        obs = result.observation
        self.reward = obs.reward
        self.done = obs.done
        self.last_components = obs.reward_components
        return f'WITNESS: {obs.witness_response}\n[{obs.questions_remaining} questions left]'

    def present_evidence(self, exhibit_id: str) -> str:
        """Present an exhibit to the witness.

        Args:
            exhibit_id: The id of the exhibit to present.

        Returns:
            The witness reaction.
        """
        if self.done:
            raise ValueError('Episode is over.')
        self.actions.append('present_evidence')
        result = self.client.step(CounselAction(tool='present_evidence', exhibit_id=exhibit_id))
        obs = result.observation
        self.reward = obs.reward
        self.done = obs.done
        self.last_components = obs.reward_components
        return obs.witness_response

    def rest_case(self) -> str:
        """End the cross-examination.

        Returns:
            Final episode status.
        """
        self.actions.append('rest_case')
        result = self.client.step(CounselAction(tool='rest_case'))
        obs = result.observation
        self.reward = obs.reward
        self.done = True
        self.last_components = obs.reward_components
        return f'Case rested. Final reward: {self.reward:.3f}'

    @property
    def action_diversity(self) -> int:
        return len(set(self.actions))

    @property
    def unique_question_patterns_pct(self) -> float:
        patterns = {question_pattern(q) for q in self.questions}
        return len(patterns) / max(1, len(self.questions))

In [ ]:
def reward_func(environments, **kwargs) -> List[float]:
    # The environment computes WeightedSum(primary + auxiliary).
    return [float(env.reward) for env in environments]

def log_diversity(environments: List[CounselToolEnv], path: str = 'rollout_debug.jsonl') -> None:
    rows = []
    for idx, env in enumerate(environments):
        row = {
            'episode': idx,
            'stage': env.stage,
            'reward': env.reward,
            'action_diversity': env.action_diversity,
            'unique_question_patterns_pct': env.unique_question_patterns_pct,
            **env.last_components,
        }
        rows.append(row)
    with open(path, 'a', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(row, sort_keys=True) + '\n')

# Failure detection required by the hackathon brief.
def warn_on_reward_difficulty(rewards: List[float]) -> None:
    avg_reward = float(np.mean(rewards)) if rewards else 0.0
    if avg_reward == 0.0:
        print('WARNING: ENV TOO HARD')
    if avg_reward > 0.7:
        print('WARNING: ENV TOO EASY')
    print(f'avg_reward={avg_reward:.3f}')

In [ ]:
training_args = GRPOConfig(
    use_vllm=True,
    vllm_mode='colocate',
    chat_template_kwargs={'enable_thinking': False},
    max_completion_length=1024,
    num_generations=6,
    temperature=0.8,
    top_p=0.95,
    gradient_accumulation_steps=16,
    per_device_train_batch_size=1,
    learning_rate=1e-5,
    num_train_epochs=1,
    max_steps=MAX_STEPS,
    beta=0.04,
    log_completions=True,
    logging_steps=5,
    report_to='none',
    output_dir=OUTPUT_DIR,
    push_to_hub=False,
)

trainer = None
if RUN_TRAINING:
    trainer = GRPOTrainer(
        model=MODEL,
        reward_funcs=reward_func,
        train_dataset=dataset,
        args=training_args,
        environment_factory=CounselToolEnv,
    )
    train_result = trainer.train()
    trainer.save_model(OUTPUT_DIR)
else:
    print('Training is disabled. Set RUN_TRAINING=1 only after credit approval.')
    print('Tiny dry-run example after approval: RUN_TRAINING=1 COUNSEL_MODEL=Qwen/Qwen3-0.6B COUNSEL_MAX_STEPS=5 COUNSEL_DATASET_SIZE=12')

In [ ]:
# Local, no-credit rollout diagnostics. This uses the in-process env, not HF.
from counsel_env.server.diagnostics import run_rollout_diagnostics

rows = run_rollout_diagnostics(output_path='rollout_debug.jsonl', num_episodes=10, policy='mixed')
rewards = [row['total_reward'] for row in rows]
warn_on_reward_difficulty(rewards)
print(f'non_zero_rollouts={sum(r > 0 for r in rewards)}/{len(rewards)}')

In [ ]:
# Plot reward trend after a real run. This cell expects trainer logs or an exported metrics list.
# It intentionally does not fabricate a reward curve.
import matplotlib.pyplot as plt

metrics_path = Path(OUTPUT_DIR) / 'trainer_state.json'
if metrics_path.exists():
    state = json.loads(metrics_path.read_text())
    history = state.get('log_history', [])
    steps = [row['step'] for row in history if 'reward' in row]
    rewards = [row['reward'] for row in history if 'reward' in row]
    if rewards:
        plt.figure(figsize=(7, 4))
        plt.plot(steps, rewards, marker='o')
        plt.xlabel('training step')
        plt.ylabel('GRPO reward')
        plt.title('Counsel-Env reward trend')
        plt.grid(alpha=0.3)
        plt.savefig('reward_curve.png', dpi=160, bbox_inches='tight')
        plt.show()
else:
    print('No trainer_state.json yet. Run approved training first, then rerun this cell.')